In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)
print("✅ Libraries imported")

✅ Libraries imported


In [2]:
TOTAL_ORDERS = 30000
BEFORE_DAYS  = 30
AFTER_DAYS   = 30
START_DATE   = datetime(2024, 1, 1)
CUTOFF_DATE  = START_DATE + timedelta(days=BEFORE_DAYS)
END_DATE     = CUTOFF_DATE + timedelta(days=AFTER_DAYS)

print(f"✅ Config set")
print(f"   Start date  : {START_DATE.date()}")
print(f"   Cutoff date : {CUTOFF_DATE.date()}")
print(f"   End date    : {END_DATE.date()}")
print(f"   Total orders: {TOTAL_ORDERS:,}")

✅ Config set
   Start date  : 2024-01-01
   Cutoff date : 2024-01-31
   End date    : 2024-03-01
   Total orders: 30,000


In [3]:
def get_peak_flag(hour):
    if hour in range(12, 15) or hour in range(19, 23):
        return "Peak"
    return "Non-Peak"

def get_distance_bucket(km):
    if km <= 3:
        return "Short (0-3 km)"
    elif km <= 7:
        return "Medium (3-7 km)"
    else:
        return "Long (7+ km)"

def generate_delivery_time(distance_km, peak, feature_active, zone):
    base_time    = 10 + (distance_km * 2.5)
    peak_penalty = np.random.normal(5, 2) if peak == "Peak" else np.random.normal(1, 0.5)
    zone_penalty = {"High": 0, "Medium": 2, "Low": 5}[zone]

    if not feature_active:
        feature_impact = np.random.normal(6, 2.5)
        if peak == "Peak":
            feature_impact *= 1.8
        if zone == "Low":
            feature_impact *= 1.3
    else:
        feature_impact = 0

    total = base_time + peak_penalty + zone_penalty + feature_impact
    noise = np.random.normal(0, 1.5)
    return round(max(5, total + noise), 1)

def generate_promised_eta(distance_km):
    return round(10 + (distance_km * 2.2) + np.random.uniform(0, 3), 1)

def get_cancellation_prob(delivery_time, promised_eta, feature_active):
    delay     = delivery_time - promised_eta
    base_prob = 0.03
    if delay > 15:
        base_prob += 0.12
    elif delay > 8:
        base_prob += 0.05
    if not feature_active:
        base_prob += 0.02
    return min(base_prob, 0.25)

print("✅ Helper functions defined")

✅ Helper functions defined


In [4]:
records  = []
order_id = 1000

for _ in range(TOTAL_ORDERS):
    random_day   = random.randint(0, BEFORE_DAYS + AFTER_DAYS - 1)
    order_date   = START_DATE + timedelta(days=random_day)
    order_hour   = random.choices(
        range(8, 24),
        weights=[1,1,2,3,4,4,3,2,3,4,4,3,2,2,1,1],
        k=1
    )[0]
    order_minute = random.randint(0, 59)
    order_time   = order_date.replace(hour=order_hour, minute=order_minute)

    feature_active = order_date < CUTOFF_DATE
    zone           = random.choices(["High", "Medium", "Low"], weights=[50, 30, 20])[0]
    distance_km    = round(np.random.lognormal(mean=1.5, sigma=0.5), 2)
    distance_km    = min(distance_km, 20)
    peak_flag      = get_peak_flag(order_hour)

    delivery_time  = generate_delivery_time(distance_km, peak_flag, feature_active, zone)
    promised_eta   = generate_promised_eta(distance_km)
    cancel_prob    = get_cancellation_prob(delivery_time, promised_eta, feature_active)

    order_status   = "Cancelled" if random.random() < cancel_prob else "Delivered"
    customer_rating = (
        None if order_status == "Cancelled"
        else round(max(1, min(5, np.random.normal(
            4.2 if feature_active else 3.7, 0.6
        ))), 1)
    )

    records.append({
        "order_id"             : order_id,
        "order_date"           : order_date.strftime("%Y-%m-%d"),
        "order_time"           : order_time.strftime("%Y-%m-%d %H:%M:%S"),
        "order_hour"           : order_hour,
        "feature_active"       : "Yes" if feature_active else "No",
        "period"               : "Before" if feature_active else "After",
        "zone"                 : zone,
        "delivery_distance_km" : distance_km,
        "distance_bucket"      : get_distance_bucket(distance_km),
        "peak_flag"            : peak_flag,
        "delivery_time_min"    : delivery_time,
        "promised_eta_min"     : promised_eta,
        "order_status"         : order_status,
        "customer_rating"      : customer_rating,
        "rider_id"             : f"R{random.randint(1, 200):03d}",
    })

    order_id += 1

df = pd.DataFrame(records)
print(f"✅ Dataset generated")
print(f"   Shape  : {df.shape}")
print(f"   Periods: {df['period'].value_counts().to_dict()}")
df.head()

✅ Dataset generated
   Shape  : (30000, 15)
   Periods: {'Before': 15009, 'After': 14991}


,order_id,order_date,order_time,order_hour,feature_active,period,zone,delivery_distance_km,distance_bucket,peak_flag,delivery_time_min,promised_eta_min,order_status,customer_rating,rider_id
0,1000,2024-02-10,2024-02-10 11:47:00,11,No,After,High,5.75,Medium (3-7 km),Non-Peak,35.2,23.1,Delivered,3.9,R189
1,1001,2024-01-07,2024-01-07 18:57:00,18,Yes,Before,Medium,7.43,Long (7+ km),Non-Peak,30.5,26.9,Delivered,4.3,R009
2,1002,2024-01-02,2024-01-02 10:14:00,10,Yes,Before,Medium,1.72,Short (0-3 km),Non-Peak,15.6,15.6,Cancelled,NaN,R051
3,1003,2024-02-15,2024-02-15 17:34:00,17,No,After,High,3.62,Medium (3-7 km),Non-Peak,20.7,20.3,Delivered,3.7,R072
4,1004,2024-02-21,2024-02-21 20:48:00,20,No,After,Low,2.20,Short (0-3 km),Peak,37.4,15.0,Delivered,3.9,R088


In [5]:
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/raw_orders.csv", index=False)
print(f"✅ Saved: ../data/raw_orders.csv")
print(f"   Total rows: {len(df):,}")

✅ Saved: ../data/raw_orders.csv
   Total rows: 30,000
